# Canonical variable candidate check

Select any of the six canonical variables, inspect candidate coverage, apply its plausible-value rule, and draw time-series and ECDF figures.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sqlalchemy as sa
from IPython.display import display

HERE = Path.cwd()
sys.path.insert(0, str(HERE / 'func'))
from func_variable_mapping import *

DB_URL = os.getenv(
    'CRMP_DB_URL',
    'postgresql://tongli1997@proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca/crmp'
    '?keepalives=1&keepalives_idle=300&keepalives_interval=300'
    '&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass',
)
engine = sa.create_engine(DB_URL, pool_pre_ping=True)
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Parameters

Change `CANONICAL_VARIABLE` to any value shown in `CANONICAL_VARIABLES`.
`STATION_LIMIT` is the maximum number of stations to randomly select without replacement.
Use `RANDOM_SEED = None` for a fresh sample each time you rerun the station-selection cell,
or set an integer (for example, `42`) for a repeatable sample.


In [ ]:
# Choose the variable, sample size, and optional random seed.
CANONICAL_VARIABLE = 'daily_min_temperature'
PLOT_VARIABLE_IDS = None  # For example: [434, 721]; None plots all candidates.
NETWORK_ID = 2  # Change only this value to work on another network.
NETWORK_DIR = HERE / 'networks' / f'network_{NETWORK_ID}'
NETWORK_DIR.mkdir(parents=True, exist_ok=True)
STATION_LIMIT = 10
RANDOM_SEED = None  # Set an integer for a repeatable sample.
FIGURE_DIR = NETWORK_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print('Current variable:', CANONICAL_VARIABLE, sep='\n* ')

print('-----' * 10)
print('Available variables:')
print(*CANONICAL_VARIABLES, sep='\n')

if CANONICAL_VARIABLE not in CANONICAL_VARIABLES:
    raise ValueError(f'Choose one of: {CANONICAL_VARIABLES}')

## 2. Count matching stations, randomly select stations, and load data

The total counts stations in this network containing observations for **every** candidate
variable listed below (before applying the plausible-value rule). If fewer than
`STATION_LIMIT` stations match, all matching stations are selected.


In [ ]:
# Load the persistent registry directly from CSV.
REGISTRY_PATH = NETWORK_DIR / 'candidate_mappings.csv'
all_candidate_mappings = pd.read_csv(REGISTRY_PATH)
candidate_mappings = all_candidate_mappings[
    all_candidate_mappings['network_id'] == NETWORK_ID
].copy()
if candidate_mappings.empty:
    raise ValueError(
        f'No saved mappings for network {NETWORK_ID} in {REGISTRY_PATH}')

candidate_rows = candidate_mappings[
    candidate_mappings['canonical_variable'] == CANONICAL_VARIABLE
]
if candidate_rows.empty:
    raise ValueError(
        f'No {CANONICAL_VARIABLE} candidates saved for network {NETWORK_ID}'
    )
candidate_ids = candidate_rows['vars_id'].tolist()
display(candidate_rows)

if not isinstance(STATION_LIMIT, int) or isinstance(STATION_LIMIT, bool) or STATION_LIMIT < 1:
    raise ValueError('STATION_LIMIT must be a positive integer.')

eligible_stations = select_candidate_stations(
    engine, candidate_ids, network_id=NETWORK_ID, limit=None
)
total_matching_stations = len(eligible_stations)
print(f'Stations containing all {len(candidate_ids)} candidate variables: '
      f'{total_matching_stations}')
if eligible_stations.empty:
    raise ValueError(
        'No stations contain all candidate variables in this network.')

comparison_stations = eligible_stations.sample(
    n=min(STATION_LIMIT, total_matching_stations),
    replace=False,
    random_state=RANDOM_SEED,
).sort_values('station_id').reset_index(drop=True)
print(f'Randomly selected {len(comparison_stations)} of '
      f'{total_matching_stations} matching stations.')
comparison_station_ids = comparison_stations['station_id'].tolist()
display(comparison_stations)

candidate_profile = profile_candidate_variables(
    engine, candidate_ids, network_id=NETWORK_ID,
    station_ids=comparison_station_ids
)
candidate_summary = summarize_candidate_profiles(candidate_profile)
display(candidate_summary.round(3))
display(candidate_profile)

## 3. Apply the canonical plausible-value rule

In [ ]:
observations = fetch_candidate_observations(
    engine, candidate_ids, comparison_station_ids
)
clean_data, qc_summary = apply_canonical_rule(
    observations, CANONICAL_VARIABLE
)
display(qc_summary.round(2))

## 4. Time series

In [ ]:
# PLOT_VARIABLE_IDS = [442, 726]  # For example: [434, 721]; None plots all candidates.

In [ ]:
unit_labels = {
    'celsius': 'Temperature (°C)', 'mm': 'Amount (mm)', 'cm': 'Amount (cm)'}
ylabel = unit_labels[CANONICAL_RULES[CANONICAL_VARIABLE].unit]
plot_data = clean_data
if PLOT_VARIABLE_IDS is not None:
    plot_data = clean_data[
        clean_data['vars_id'].isin(PLOT_VARIABLE_IDS)
    ].copy()
    if plot_data.empty:
        raise ValueError(
            'None of PLOT_VARIABLE_IDS has valid data for the selected stations.'
        )

stations = list(plot_data.groupby('station_id'))

fig, axes = plt.subplots(len(stations), 1, figsize=(14, 4 * len(stations)),
                         sharex=True, sharey=True, squeeze=False)
for ax, (station_id, station_data) in zip(axes.ravel(), stations):
    for (vars_id, variable_name), variable_data in station_data.groupby(
        ['vars_id', 'net_var_name']
    ):
        ax.plot(variable_data['obs_time'], variable_data['datum'],
                linewidth=1, alpha=0.8,
                label=f'{vars_id} — {variable_name}')
    ax.set_title(
        f"{station_data['station_name'].iloc[0]} (station {station_id})")
    ax.set_ylabel(ylabel)
    ax.legend()
axes[-1, 0].set_xlabel('Time')
fig.suptitle(f'{CANONICAL_VARIABLE}: candidate time series', y=1.002)
plt.tight_layout()
time_series_path = FIGURE_DIR / (
    f'network_{NETWORK_ID}_{CANONICAL_VARIABLE}_time_series.png'
)
fig.savefig(time_series_path, dpi=150, bbox_inches='tight')
print(f'Saved: {time_series_path}')
plt.show()

## 5. Empirical cumulative distributions

Precipitation and snowfall ECDFs use positive values only and a logarithmic amount axis. Temperature and snow-depth ECDFs retain all plausible values.

In [ ]:
positive_only = CANONICAL_VARIABLE in {
    'precipitation_amount', 'snowfall_amount'}
log_x = positive_only
ncols = 2
nrows = int(np.ceil(len(stations) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows),
                         sharex=True, sharey=True, squeeze=False)
axes = axes.ravel()

for ax, (station_id, station_data) in zip(axes, stations):
    for (vars_id, variable_name), variable_data in station_data.groupby(
        ['vars_id', 'net_var_name']
    ):
        values = variable_data['datum'].dropna()
        if positive_only:
            values = values[values > 0]
        values = np.sort(values.to_numpy())
        if len(values):
            cumulative = np.arange(1, len(values) + 1) / len(values)
            ax.step(values, cumulative, where='post',
                    label=f'{vars_id} — {variable_name}')
    if log_x:
        ax.set_xscale('log')
    ax.set_title(
        f"{station_data['station_name'].iloc[0]} (station {station_id})")
    ax.set_xlabel(
        ylabel + (' — positive values, log scale' if positive_only else ''))
    ax.set_ylabel('Cumulative proportion')
    ax.legend()

for ax in axes[len(stations):]:
    ax.set_visible(False)
fig.suptitle(f'{CANONICAL_VARIABLE}: candidate ECDFs', y=1.002)
plt.tight_layout()
cdf_path = FIGURE_DIR / (
    f'network_{NETWORK_ID}_{CANONICAL_VARIABLE}_cdf.png'
)
fig.savefig(cdf_path, dpi=150, bbox_inches='tight')
print(f'Saved: {cdf_path}')
plt.show()

## 6. Choose final source IDs using a station sample

Set `SOURCE_STATION_LIMIT = 20` below to check up to 20 randomly selected stations
that contain **every candidate variable** (both variables when there are two).
The eligible station list is reused from section 2; rerun section 2 after changing
the network or canonical variable. This avoids repeating its eligibility query.
The sample is drawn independently of the plot sample. Set `SOURCE_RANDOM_SEED`
to an integer for repeatable sampling, or `None` for a fresh sample. If fewer than
20 eligible stations exist, all are checked.

Eligibility means at least one raw observation for each candidate at any date;
it does not require non-null values, recent data, or matching observation times.
Consequently, the redundant `has_observations` column is omitted.
The per-station table records `latest_observation_from_cutoff`, the latest non-null
observation on or after `AUTO_KEEP_FROM` (January 1, 2026, inclusive), and whether
such an observation exists. No plausible-value filtering is applied.
The summary table shows `total_station_count` and `recent_non_null_station_count`
**within this sample only**, counting a station once per candidate. These are not
network totals. `total_station_count` is the actual sample size for every candidate.
A missing recent observation in this sample does not establish
that the variable has no recent data elsewhere in the network.

A candidate with recent non-null data in the sample is automatically kept. Edit
`MANUAL_SOURCE_IDS` in the following cell to keep additional candidates. Rerun the
manual cell after changing the sample, variable, or cutoff, then run the save cell.

The CSV saves the sampled station IDs, actual sample size, and sampling settings.
Detailed station conditions and coverage counts are displayed above rather than
stored in the final CSV. `recent_source_vars_ids` and
`recent_source_latest_observations` refer only to this sample. `auto_keep_from`
records the cutoff; `source_checked_at_utc` is the single run timestamp, recording
when the sample check finished in ISO 8601 UTC. Saving again preserves that check
time. The save cell also removes the older condition, coverage, and save-time
columns from existing CSV rows.

Tables are organized by `vars_id`: the detailed table lists stations within each
variable ID, and the summary shows one candidate row indexed by variable ID.


In [ ]:
AUTO_KEEP_FROM = '2026-01-01'

SOURCE_STATION_LIMIT = 20
SOURCE_RANDOM_SEED = None  # Integer for repeatable sampling.
if (not isinstance(SOURCE_STATION_LIMIT, int)
        or isinstance(SOURCE_STATION_LIMIT, bool) or SOURCE_STATION_LIMIT < 1):
    raise ValueError('SOURCE_STATION_LIMIT must be a positive integer.')

# Reuse section 2's stations containing every candidate variable.
# No second network-wide eligibility query is needed here.
if eligible_stations.empty:
    raise ValueError(
        'No stations contain every candidate variable. Rerun section 2.')
source_station_ids = sorted(int(value) for value in eligible_stations.sample(
    n=min(SOURCE_STATION_LIMIT, len(eligible_stations)),
    random_state=SOURCE_RANDOM_SEED, replace=False,
)['station_id'])

station_condition_query = sa.text("""
    SELECT s.station_id, v.vars_id,
           latest.obs_time AS latest_observation_from_cutoff
    FROM meta_station s
    CROSS JOIN meta_vars v
    LEFT JOIN LATERAL (
        SELECT o.obs_time FROM obs_raw o
        JOIN meta_history h USING (history_id)
        WHERE h.station_id = s.station_id AND o.vars_id = v.vars_id
          AND o.datum IS NOT NULL AND o.obs_time >= :auto_keep_from
        ORDER BY o.obs_time DESC LIMIT 1
    ) latest ON TRUE
    WHERE s.network_id = :network_id AND s.station_id IN :station_ids
      AND v.vars_id IN :vars_ids
    ORDER BY s.station_id, v.vars_id
""").bindparams(
    sa.bindparam('vars_ids', expanding=True),
    sa.bindparam('station_ids', expanding=True),
)
source_station_conditions = pd.read_sql(
    station_condition_query, engine,
    params={
        'vars_ids': [int(value) for value in candidate_ids],
        'station_ids': source_station_ids,
        'network_id': NETWORK_ID,
        'auto_keep_from': pd.Timestamp(AUTO_KEEP_FROM).to_pydatetime(),
    },
)
source_station_conditions['latest_observation_from_cutoff'] = pd.to_datetime(
    source_station_conditions['latest_observation_from_cutoff']
)
source_station_conditions['has_recent_non_null_observations'] = (
    source_station_conditions['latest_observation_from_cutoff'].notna()
)
station_coverage = source_station_conditions.groupby('vars_id', as_index=False).agg(
    total_station_count=('station_id', 'nunique'),
    recent_non_null_station_count=('has_recent_non_null_observations', 'sum'),
    latest_observation=('latest_observation_from_cutoff', 'max'),
)
source_checked_at_utc = pd.Timestamp.now(tz='UTC').isoformat()
print(
    f'Checked {len(source_station_ids)} sampled stations: {source_station_ids}')
display(source_station_conditions.set_index(
    ['vars_id', 'station_id']).sort_index())
recent_candidates = station_coverage.loc[
    station_coverage['recent_non_null_station_count'] > 0
]
AUTO_SOURCE_IDS = sorted(int(value) for value in recent_candidates['vars_id'])
recent_source_latest_observations = {
    str(int(row.vars_id)): pd.Timestamp(row.latest_observation).isoformat()
    for row in recent_candidates.itertuples(index=False)
}
# Prevent a stale automatic selection from being saved for another variable/network.
auto_selection_context = (
    NETWORK_ID, CANONICAL_VARIABLE, tuple(
        sorted(candidate_ids)), AUTO_KEEP_FROM, tuple(source_station_ids)
)
source_review = candidate_rows.merge(
    station_coverage[['vars_id', 'total_station_count',
                      'recent_non_null_station_count']],
    on='vars_id', how='left', validate='many_to_one',
)
count_columns = ['total_station_count', 'recent_non_null_station_count']
source_review[count_columns] = source_review[count_columns].fillna(
    0).astype(int)
source_review['latest_observation_from_cutoff'] = source_review['vars_id'].map(
    lambda value: recent_source_latest_observations.get(str(int(value)), pd.NA)
)
source_review['decision'] = source_review['vars_id'].map(
    lambda value: 'automatic keep' if value in AUTO_SOURCE_IDS else 'manual decision'
)
print(f'Automatically kept in the sample (non-null observations from {AUTO_KEEP_FROM} onward): '
      f'{AUTO_SOURCE_IDS}')
display(source_review.sort_values('vars_id').set_index('vars_id'))
# print('Candidates available for your manual selection:')
# display(source_review.loc[source_review['decision'] == 'manual decision'])

In [ ]:
# Edit this list to keep additional candidates after reviewing the plots.
# Reset/review it whenever you change CANONICAL_VARIABLE.
MANUAL_SOURCE_IDS = [435]  # Example: [435]
manual_selection_context = (
    NETWORK_ID, CANONICAL_VARIABLE, tuple(
        sorted(candidate_ids)), AUTO_KEEP_FROM, tuple(source_station_ids)
)

In [ ]:
# Record the reviewed selection and merge policy; do not combine data here.
import json
current_selection_context = (
    NETWORK_ID, CANONICAL_VARIABLE, tuple(
        sorted(candidate_ids)), AUTO_KEEP_FROM, tuple(source_station_ids)
)
if (auto_selection_context != current_selection_context
        or manual_selection_context != current_selection_context):
    raise ValueError('Variable, network, candidates, cutoff, or sample changed. '
                     'Rerun the automatic and manual selection cells.')
FINAL_SOURCE_IDS = sorted(set(AUTO_SOURCE_IDS) | set(MANUAL_SOURCE_IDS))
print('Automatic IDs:', AUTO_SOURCE_IDS)
print('Manual IDs:', MANUAL_SOURCE_IDS)
print('Final source IDs:', FINAL_SOURCE_IDS)
FINAL_RULE_PATH = NETWORK_DIR / 'final_variable_rules.csv'

if not FINAL_SOURCE_IDS:
    print(
        f'No rule saved for {CANONICAL_VARIABLE}: review the figures, '
        'fill MANUAL_SOURCE_IDS above, then rerun this cell.'
    )
else:
    unknown_ids = set(FINAL_SOURCE_IDS) - set(candidate_ids)
    if unknown_ids:
        raise ValueError(
            f'FINAL_SOURCE_IDS are not candidates for '
            f'{CANONICAL_VARIABLE}: {sorted(unknown_ids)}'
        )

    final_rule = pd.DataFrame([{
        'network_id': NETWORK_ID,
        'canonical_variable': CANONICAL_VARIABLE,
        'source_vars_ids': json.dumps(FINAL_SOURCE_IDS),
        'auto_keep_from': AUTO_KEEP_FROM,
        'source_check_scope': 'random_stations_with_all_candidates',
        'source_station_limit': SOURCE_STATION_LIMIT,
        'source_random_seed': SOURCE_RANDOM_SEED,
        'source_station_count': len(source_station_ids),
        'source_station_ids': json.dumps(source_station_ids),
        'source_checked_at_utc': source_checked_at_utc,
        'recent_source_vars_ids': json.dumps(AUTO_SOURCE_IDS),
        'recent_source_latest_observations': json.dumps(
            recent_source_latest_observations, sort_keys=True
        ),
        'time_match': 'exact_timestamp',
        'overlap_method': 'mean',
        'single_source_method': 'use_available',
    }])

    if FINAL_RULE_PATH.exists():
        saved_rules = pd.read_csv(FINAL_RULE_PATH).drop(
            columns=['source_station_coverage', 'source_station_conditions',
                     'rule_saved_at_utc'],
            errors='ignore',
        )
        same_mapping = (
            (saved_rules['network_id'] == NETWORK_ID)
            & (saved_rules['canonical_variable'] == CANONICAL_VARIABLE)
        )
        saved_rules = saved_rules.loc[~same_mapping]
        final_rules = pd.concat(
            [saved_rules, final_rule], ignore_index=True
        )
    else:
        final_rules = final_rule

    final_rules = final_rules.sort_values(
        ['network_id', 'canonical_variable']
    ).reset_index(drop=True)
    final_rules.to_csv(FINAL_RULE_PATH, index=False)
    print(f'Saved rule: {NETWORK_ID} / {CANONICAL_VARIABLE}')
    display(final_rules)